In [1]:
from elevant.llm_client import LLMClient
from elevant.linkers.graph_linker import GraphLinker
from elevant.models.entity_database import EntityDatabase
from elevant import settings
from elevant.models.article import Article, article_from_json
from elevant.linkers.linking_system import LinkingSystem

# client = LLMClient('Qwen/Qwen3-8B')
# client.call([{"role": "user", "content": "What is machine learning?"}])

custom_db = True
entity_db = EntityDatabase()
if custom_db:
    entity_db.load_custom_entity_names(settings.CUSTOM_ENTITY_TO_NAME_FILE)
    entity_db.load_custom_entity_types(settings.CUSTOM_ENTITY_TO_TYPES_FILE)
    entity_db.load_custom_entity_descriptions(settings.CUSTOM_ENTITY_TO_DESCRIPTIONS_FILE)
else:
    entity_db.load_entity_names()
    entity_db.load_entity_types()


config = {
  "linker_name": "Graph LLM",
  "llm_model_path": "Qwen/Qwen2.5-7B-Instruct",
  "n_descriptions": 3,
  "k_search": 5,
  "t_max": 5,
  "high_confidence_threshold": 0.9,
  "experiment_description": "Optimized graph-based entity linking with batch processing. 5-10x faster than previous version."
}

# llm_client = LLMClient('Qwen/Qwen3-8B')
# output = llm_client.call([{"role": "user", "content": "What is machine learning?"}])

/media/volume/LLMRag2/miniconda3/envs/running/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/media/volume/LLMRag2/miniconda3/envs/running/lib/python3.12/site-packages/transformers/utils/hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


In [2]:
linker = LinkingSystem(
    linker_name="graph-llm",
    config_path='/media/volume/LLMRag2/.local/ActDiseaseEL/configs/graph-llm.config.json',
    coref_linker=None,
    min_score=0,
    type_mapping_file=settings.QID_TO_WHITELIST_TYPES_DB,
    custom_kb=True)

Loading checkpoint shards: 100%|██████████| 4/4 [00:17<00:00,  4.43s/it]


In [3]:
json_str = '{"id": 324, "title": "BMJ_1929_03_16_vol001_nr3558_art044_pmc2450118.txt_line_170", "text": "The patient presented with T2DM and hypertension. Blood glucose was elevated."}'

article = article_from_json(json_str)
article

{'id': 324, 'title': 'BMJ_1929_03_16_vol001_nr3558_art044_pmc2450118.txt_line_170', 'text': 'The patient presented with T2DM and hypertension. Blood glucose was elevated.', 'evaluation_span': (0, 77), 'labels': []}

In [4]:
# entity_db.get_candidates('type 2 diabetes mellitus')
# entity_db.get_entity_description('DOID:10763')
# a = ['<|im_start|>system\nYou are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>\n<|im_start|>user\nKNOWLEDGE BASE: Human Disease Ontology (DOID)\nTASK: Link Medical Mention to DOID Disease\n\n=== MEDICAL MENTION ===\nMention: T2DM\nContext: The patient presented with ###T2DM### and hypertension. Blood glucose was elevated.\n\n=== CANDIDATE DISEASES ===\n1. type 2 diabetes mellitus (ID: DOID:9352)\n   Description: A diabetes mellitus that is characterized by high blood sugar, insulin resistance, and relative lack\n\n=== YOUR TASK ===\nSelect the disease that best matches the mention in context.\n\nCRITERIA:\n1. Name match: Does the name match the medical term?\n2. Specificity: Is it the right level of detail?\n3. Context fit: Does it fit the clinical context?\n\n=== REQUIRED OUTPUT FORMAT ===\nYou MUST output in this EXACT format (both fields required):\nENTITY ID: [candidate_id_from_list_above]\nCONFIDENCE: [confidence_score_between_0.0_and_1.0]\n\nWhere:\n- ENTITY ID: The exact ID from the candidate list (e.g., "DOID:12345")\n- CONFIDENCE: A number between 0.0 and 1.0 indicating how confident you are:\n  * 0.9-1.0: Very high confidence (exact match, clear context)\n  * 0.7-0.9: High confidence (good match, some ambiguity)\n  * 0.5-0.7: Medium confidence (partial match, some uncertainty)\n  * 0.0-0.5: Low confidence (weak match, high uncertainty)\n\n=== CRITICAL: FORMAT EXAMPLE ===\nIf candidate #2 is the best match and you\'re very confident:\nENTITY ID: DOID:12345\nCONFIDENCE: 0.95\n\nIf candidate #1 is a good match but you\'re moderately confident:\nENTITY ID: DOID:67890\nCONFIDENCE: 0.75\n\nIf no candidate matches well:\nENTITY ID: <NIL>\nCONFIDENCE: 0.2\n\n=== STRICT REQUIREMENTS ===\n1. MUST output "ENTITY ID: " followed by the candidate ID or "<NIL>"\n2. MUST output "CONFIDENCE: " followed by a number between 0.0 and 1.0\n3. Both lines are REQUIRED\n4. Use exact candidate ID from the list above\n5. Confidence must be a valid float between 0.0 and 1.0\n6. NO additional text, NO explanations, ONLY these two lines\n\n=== OUTPUT NOW ===\n<|im_end|>\n<|im_start|>assistant\n', '<|im_start|>system\nYou are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>\n<|im_start|>user\nKNOWLEDGE BASE: Human Disease Ontology (DOID)\nTASK: Link Medical Mention to DOID Disease\n\n=== MEDICAL MENTION ===\nMention: hypertension\nContext: The patient presented with T2DM and ###hypertension### . Blood glucose was elevated.\n\n=== CANDIDATE DISEASES ===\n1. hypertension (ID: DOID:10763)\n   Description: An artery disease characterized by chronic elevated blood pressure in the arteries.\n\n=== YOUR TASK ===\nSelect the disease that best matches the mention in context.\n\nCRITERIA:\n1. Name match: Does the name match the medical term?\n2. Specificity: Is it the right level of detail?\n3. Context fit: Does it fit the clinical context?\n\n=== REQUIRED OUTPUT FORMAT ===\nYou MUST output in this EXACT format (both fields required):\nENTITY ID: [candidate_id_from_list_above]\nCONFIDENCE: [confidence_score_between_0.0_and_1.0]\n\nWhere:\n- ENTITY ID: The exact ID from the candidate list (e.g., "DOID:12345")\n- CONFIDENCE: A number between 0.0 and 1.0 indicating how confident you are:\n  * 0.9-1.0: Very high confidence (exact match, clear context)\n  * 0.7-0.9: High confidence (good match, some ambiguity)\n  * 0.5-0.7: Medium confidence (partial match, some uncertainty)\n  * 0.0-0.5: Low confidence (weak match, high uncertainty)\n\n=== CRITICAL: FORMAT EXAMPLE ===\nIf candidate #2 is the best match and you\'re very confident:\nENTITY ID: DOID:12345\nCONFIDENCE: 0.95\n\nIf candidate #1 is a good match but you\'re moderately confident:\nENTITY ID: DOID:67890\nCONFIDENCE: 0.75\n\nIf no candidate matches well:\nENTITY ID: <NIL>\nCONFIDENCE: 0.2\n\n=== STRICT REQUIREMENTS ===\n1. MUST output "ENTITY ID: " followed by the candidate ID or "<NIL>"\n2. MUST output "CONFIDENCE: " followed by a number between 0.0 and 1.0\n3. Both lines are REQUIRED\n4. Use exact candidate ID from the list above\n5. Confidence must be a valid float between 0.0 and 1.0\n6. NO additional text, NO explanations, ONLY these two lines\n\n=== OUTPUT NOW ===\n<|im_end|>\n<|im_start|>assistant\n']
# print(a[0])

In [5]:
output = linker.linker.predict(article.text)

In [6]:
output

{(27,
  31): <elevant.models.entity_prediction.EntityPrediction at 0x7494dfbe2060>,
 (36,
  48): <elevant.models.entity_prediction.EntityPrediction at 0x7494dfb5d940>}

In [7]:
a = ['<|im_start|>system\nYou are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>\n<|im_start|>user\nKNOWLEDGE BASE: Human Disease Ontology (DOID)\nTASK: Link Medical Mention to DOID Disease\n\n=== MEDICAL MENTION ===\nMention: T2DM\nContext: The patient presented with ###T2DM### and hypertension. Blood glucose was elevated.\n\n=== CANDIDATE DISEASES ===\n1. type 2 diabetes mellitus (ID: DOID:9352)\n   Description: A diabetes mellitus that is characterized by high blood sugar, insulin resistance, and relative lack\n\n=== YOUR TASK ===\nSelect the disease that best matches the mention in context.\n\nCRITERIA:\n1. Name match: Does the name match the medical term?\n2. Specificity: Is it the right level of detail?\n3. Context fit: Does it fit the clinical context?\n\n=== REQUIRED OUTPUT FORMAT ===\nYou MUST output in this EXACT format (both fields required):\nENTITY ID: [candidate_id_from_list_above]\nCONFIDENCE: [confidence_score_between_0.0_and_1.0]\n\nWhere:\n- ENTITY ID: The exact ID from the candidate list (e.g., "DOID:12345")\n- CONFIDENCE: A number between 0.0 and 1.0 indicating how confident you are:\n  * 0.9-1.0: Very high confidence (exact match, clear context)\n  * 0.7-0.9: High confidence (good match, some ambiguity)\n  * 0.5-0.7: Medium confidence (partial match, some uncertainty)\n  * 0.0-0.5: Low confidence (weak match, high uncertainty)\n\n=== CRITICAL: FORMAT EXAMPLE ===\nIf candidate #2 is the best match and you\'re very confident:\nENTITY ID: DOID:12345\nCONFIDENCE: 0.95\n\nIf candidate #1 is a good match but you\'re moderately confident:\nENTITY ID: DOID:67890\nCONFIDENCE: 0.75\n\nIf no candidate matches well:\nENTITY ID: <NIL>\nCONFIDENCE: 0.2\n\n=== STRICT REQUIREMENTS ===\n1. MUST output "ENTITY ID: " followed by the candidate ID or "<NIL>"\n2. MUST output "CONFIDENCE: " followed by a number between 0.0 and 1.0\n3. Both lines are REQUIRED\n4. Use exact candidate ID from the list above\n5. Confidence must be a valid float between 0.0 and 1.0\n6. NO additional text, NO explanations, ONLY these two lines\n\n=== OUTPUT NOW ===\n<|im_end|>\n<|im_start|>assistant\n', '<|im_start|>system\nYou are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>\n<|im_start|>user\nKNOWLEDGE BASE: Human Disease Ontology (DOID)\nTASK: Link Medical Mention to DOID Disease\n\n=== MEDICAL MENTION ===\nMention: hypertension\nContext: The patient presented with T2DM and ###hypertension### . Blood glucose was elevated.\n\n=== CANDIDATE DISEASES ===\n1. hypertension (ID: DOID:10763)\n   Description: An artery disease characterized by chronic elevated blood pressure in the arteries.\n\n=== YOUR TASK ===\nSelect the disease that best matches the mention in context.\n\nCRITERIA:\n1. Name match: Does the name match the medical term?\n2. Specificity: Is it the right level of detail?\n3. Context fit: Does it fit the clinical context?\n\n=== REQUIRED OUTPUT FORMAT ===\nYou MUST output in this EXACT format (both fields required):\nENTITY ID: [candidate_id_from_list_above]\nCONFIDENCE: [confidence_score_between_0.0_and_1.0]\n\nWhere:\n- ENTITY ID: The exact ID from the candidate list (e.g., "DOID:12345")\n- CONFIDENCE: A number between 0.0 and 1.0 indicating how confident you are:\n  * 0.9-1.0: Very high confidence (exact match, clear context)\n  * 0.7-0.9: High confidence (good match, some ambiguity)\n  * 0.5-0.7: Medium confidence (partial match, some uncertainty)\n  * 0.0-0.5: Low confidence (weak match, high uncertainty)\n\n=== CRITICAL: FORMAT EXAMPLE ===\nIf candidate #2 is the best match and you\'re very confident:\nENTITY ID: DOID:12345\nCONFIDENCE: 0.95\n\nIf candidate #1 is a good match but you\'re moderately confident:\nENTITY ID: DOID:67890\nCONFIDENCE: 0.75\n\nIf no candidate matches well:\nENTITY ID: <NIL>\nCONFIDENCE: 0.2\n\n=== STRICT REQUIREMENTS ===\n1. MUST output "ENTITY ID: " followed by the candidate ID or "<NIL>"\n2. MUST output "CONFIDENCE: " followed by a number between 0.0 and 1.0\n3. Both lines are REQUIRED\n4. Use exact candidate ID from the list above\n5. Confidence must be a valid float between 0.0 and 1.0\n6. NO additional text, NO explanations, ONLY these two lines\n\n=== OUTPUT NOW ===\n<|im_end|>\n<|im_start|>assistant\n']

In [8]:
# # client = LLMClient('Qwen/Qwen3-8B')
# a = [[{'role': 'user', 'content': 'KNOWLEDGE BASE: Human Disease Ontology (DOID)\nTASK: Link Medical Mention to DOID Disease\n\n=== MEDICAL MENTION ===\nMention: T2DM\nContext: The patient presented with ###T2DM### and hypertension. Blood glucose was elevated.\n\n=== CANDIDATE DISEASES ===\n1. type 2 diabetes mellitus (ID: DOID:9352)\n   Description: A diabetes mellitus that is characterized by high blood sugar, insulin resistance, and relative lack\n\n=== YOUR TASK ===\nSelect the disease that best matches the mention in context.\n\nCRITERIA:\n1. Name match: Does the name match the medical term?\n2. Specificity: Is it the right level of detail?\n3. Context fit: Does it fit the clinical context?\n\n=== REQUIRED OUTPUT FORMAT ===\nYou MUST output in this EXACT format (both fields required):\nENTITY ID: [candidate_id_from_list_above]\nCONFIDENCE: [confidence_score_between_0.0_and_1.0]\n\nWhere:\n- ENTITY ID: The exact ID from the candidate list (e.g., "DOID:12345")\n- CONFIDENCE: A number between 0.0 and 1.0 indicating how confident you are:\n  * 0.9-1.0: Very high confidence (exact match, clear context)\n  * 0.7-0.9: High confidence (good match, some ambiguity)\n  * 0.5-0.7: Medium confidence (partial match, some uncertainty)\n  * 0.0-0.5: Low confidence (weak match, high uncertainty)\n\n=== CRITICAL: FORMAT EXAMPLE ===\nIf candidate #2 is the best match and you\'re very confident:\nENTITY ID: DOID:12345\nCONFIDENCE: 0.95\n\nIf candidate #1 is a good match but you\'re moderately confident:\nENTITY ID: DOID:67890\nCONFIDENCE: 0.75\n\nIf no candidate matches well:\nENTITY ID: <NIL>\nCONFIDENCE: 0.2\n\n=== STRICT REQUIREMENTS ===\n1. MUST output "ENTITY ID: " followed by the candidate ID or "<NIL>"\n2. MUST output "CONFIDENCE: " followed by a number between 0.0 and 1.0\n3. Both lines are REQUIRED\n4. Use exact candidate ID from the list above\n5. Confidence must be a valid float between 0.0 and 1.0\n6. NO additional text, NO explanations, ONLY these two lines\n\n=== OUTPUT NOW ===\n'}], [{'role': 'user', 'content': 'KNOWLEDGE BASE: Human Disease Ontology (DOID)\nTASK: Link Medical Mention to DOID Disease\n\n=== MEDICAL MENTION ===\nMention: hypertension\nContext: The patient presented with T2DM and ###hypertension### . Blood glucose was elevated.\n\n=== CANDIDATE DISEASES ===\n1. hypertension (ID: DOID:10763)\n   Description: An artery disease characterized by chronic elevated blood pressure in the arteries.\n\n=== YOUR TASK ===\nSelect the disease that best matches the mention in context.\n\nCRITERIA:\n1. Name match: Does the name match the medical term?\n2. Specificity: Is it the right level of detail?\n3. Context fit: Does it fit the clinical context?\n\n=== REQUIRED OUTPUT FORMAT ===\nYou MUST output in this EXACT format (both fields required):\nENTITY ID: [candidate_id_from_list_above]\nCONFIDENCE: [confidence_score_between_0.0_and_1.0]\n\nWhere:\n- ENTITY ID: The exact ID from the candidate list (e.g., "DOID:12345")\n- CONFIDENCE: A number between 0.0 and 1.0 indicating how confident you are:\n  * 0.9-1.0: Very high confidence (exact match, clear context)\n  * 0.7-0.9: High confidence (good match, some ambiguity)\n  * 0.5-0.7: Medium confidence (partial match, some uncertainty)\n  * 0.0-0.5: Low confidence (weak match, high uncertainty)\n\n=== CRITICAL: FORMAT EXAMPLE ===\nIf candidate #2 is the best match and you\'re very confident:\nENTITY ID: DOID:12345\nCONFIDENCE: 0.95\n\nIf candidate #1 is a good match but you\'re moderately confident:\nENTITY ID: DOID:67890\nCONFIDENCE: 0.75\n\nIf no candidate matches well:\nENTITY ID: <NIL>\nCONFIDENCE: 0.2\n\n=== STRICT REQUIREMENTS ===\n1. MUST output "ENTITY ID: " followed by the candidate ID or "<NIL>"\n2. MUST output "CONFIDENCE: " followed by a number between 0.0 and 1.0\n3. Both lines are REQUIRED\n4. Use exact candidate ID from the list above\n5. Confidence must be a valid float between 0.0 and 1.0\n6. NO additional text, NO explanations, ONLY these two lines\n\n=== OUTPUT NOW ===\n'}]]
# client.call_batch(a)